In [1]:
import sys, os
# Set working dir and path based on environment
if 'google.colab' in sys.modules:
  from google.colab import drive, files
  drive.mount('/content/drive')
  os.chdir('/content/drive/MyDrive/Autochtone_ingredient')
else:
  print("Not in Google Colab environment")

Not in Google Colab environment


In [2]:
from utils.data_processing import load_lexicon, add_clean_descriptors_column, remove_parentheses_text, DEFAULT_ADJECTIVES
import pandas as pd

In [3]:
merged_df = pd.read_csv('data/merged_ai_descriptors.csv')
merged_df.head()

,Nom scientifique,ai_descriptors,sub_category,db,category
0,Macrocystis pyrifera,"briny, umami, salty, oceanic, mineral, vegetal...",plant,local,plant
1,Palmaria palmata,"briny, salty, umami, oceanic, slightly sweet, ...",plant,local,plant
2,Porphyra perforata,"briny, umami, salty, mineral, oceanic, slightl...",plant,local,plant
3,Porphyra abbottae,"briny, umami, slightly sweet, mineral, oceanic...",plant,local,plant
4,Porphyra torta,"briny, oceanic, umami, mineral, slightly sweet...",plant,local,plant


# Clean GPT response

In [4]:
merged_df['ai_descriptors'] = merged_df['ai_descriptors'].apply(remove_parentheses_text)
merged_df['ai_descriptors']= merged_df['ai_descriptors'].str.replace(", ", ",")
merged_df.head()

,Nom scientifique,ai_descriptors,sub_category,db,category
0,Macrocystis pyrifera,"briny,umami,salty,oceanic,mineral,vegetal,slig...",plant,local,plant
1,Palmaria palmata,"briny,salty,umami,oceanic,slightly sweet,miner...",plant,local,plant
2,Porphyra perforata,"briny,umami,salty,mineral,oceanic,slightly swe...",plant,local,plant
3,Porphyra abbottae,"briny,umami,slightly sweet,mineral,oceanic,veg...",plant,local,plant
4,Porphyra torta,"briny,oceanic,umami,mineral,slightly sweet,veg...",plant,local,plant


In [5]:
# Text cleanup

lex = load_lexicon("data/lexicon.xlsx")  # ensures index='desc' and has column 'adj'

merged_df = add_clean_descriptors_column(
    df=merged_df,
    desc_col="ai_descriptors",
    name_col="Nom scientifique",
    lex=lex,
    adjectives=DEFAULT_ADJECTIVES,
    max_len=30,
    out_col="descriptor"
)

merged_df.head()


,Nom scientifique,ai_descriptors,sub_category,db,category,descriptor
0,Macrocystis pyrifera,"briny,umami,salty,oceanic,mineral,vegetal,slig...",plant,local,plant,"brine,umami,salt,ocean,mineral,vegetal,sweet,e..."
1,Palmaria palmata,"briny,salty,umami,oceanic,slightly sweet,miner...",plant,local,plant,"brine,salt,umami,ocean,sweet,mineral,earth,smo..."
2,Porphyra perforata,"briny,umami,salty,mineral,oceanic,slightly swe...",plant,local,plant,"brine,umami,salt,mineral,ocean,sweet,earth,sav..."
3,Porphyra abbottae,"briny,umami,slightly sweet,mineral,oceanic,veg...",plant,local,plant,"brine,umami,sweet,mineral,ocean,vegetal,salt,e..."
4,Porphyra torta,"briny,oceanic,umami,mineral,slightly sweet,veg...",plant,local,plant,"brine,ocean,umami,mineral,sweet,vegetal,earth,..."


In [6]:
descriptor_counts = (
    merged_df['descriptor']
    .dropna()                       # remove missing values
    .str.split(',')                 # split each cell into a list
    .explode()                      # make one row per descriptor
    .str.strip()                    # remove extra spaces
    .value_counts()                 # count each descriptor
)

print(descriptor_counts.head(20))

descriptor
sweet       991
earth       716
nut         455
fresh       415
bitter      366
savory      289
tang        277
delicate    265
aroma       264
grass       255
butter      243
floral      236
umami       227
cream       219
fruit       216
spicy       215
tender      192
vegetal     192
crisp       190
juice       180
Name: count, dtype: int64


In [7]:
merged_df.to_csv('data/merged_ai_descriptors_clean.csv', index=False)

# One Hot Encoding

In [8]:
# Create a copy of the dataframe for this method
df = merged_df.copy()

# The .str.get_dummies() method handles the splitting and encoding in one step.
# We just need to provide the separator character.
flavor_dummies = merged_df['descriptor'].str.get_dummies(sep=',')

# Concatenate the new dummies DataFrame with the original one.
# We drop the original 'genres' column.
df = pd.concat([df[['Nom scientifique']], flavor_dummies], axis=1)
df1 = df.copy(deep = True)

df1.to_csv('data/merged_ai_descriptors_dummies_full.csv', index=False)
df1.head()

,Nom scientifique,a type of white fish:,acid,acrid,aged,airy,alcohol,alkali,allium,almond,...,wheat,which includes sweet,wholesome,wild,wine,wintergreen,wobble,wood,yeast,zest
0,Macrocystis pyrifera,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
1,Palmaria palmata,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
2,Porphyra perforata,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
3,Porphyra abbottae,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0
4,Porphyra torta,0,0,0,0,0,0,0,0,0,...,0,0,0,0,0,0,0,0,0,0


In [9]:
# Calculate the threshold (1% of the number of rows)
threshold = len(df1) * 0.01
threshold = len(df1) * 0.01

# Get the sum of each dummy encoded column (excluding 'Nom scientifique')
dummy_sums = df1.drop(columns=['Nom scientifique']).sum()

# Identify columns where the sum is less than the threshold
columns_to_drop_based_on_threshold = dummy_sums[dummy_sums < threshold].index.tolist()

# Drop these columns from the dataframe
df1 = df1.drop(columns=columns_to_drop_based_on_threshold)

print(f"Dropped {len(columns_to_drop_based_on_threshold)} columns with less than 1% True values.")
print(f"Remaining columns: {df1.shape[1]}")

Dropped 283 columns with less than 1% True values.
Remaining columns: 132


adding categories one hot encoded

In [10]:
# adding food categories
# merged_df.head()

cat_dummies = merged_df['category'].str.get_dummies(sep=',')

## Looks like no correction is best - else it is heavily biased toward category - maybe use man made category or pca to soften the impact of categories
# # correction option 1 correct for the number of columns
# flavor_cols = df1.shape[1] - 1
# cat_cols = cat_dummies.shape[1]
# cat_correction = flavor_cols/cat_cols
# print(f"The number of times more flavor columns than category columns is : {cat_correction}")
# cat_dummies = cat_dummies.multiply(cat_correction)

# # Correction option 2: get average row sum of flavor cols
# df1_numeric = df1.drop(columns=['Nom scientifique'])
# average_row_sum = df1_numeric.sum(axis=1).mean()
# print(f"The average row sum for df1 is: {average_row_sum}")

# cat_dummies = cat_dummies.multiply(average_row_sum )

# Combine the 2 column sets
for col in cat_dummies.columns:
  cat_col = "cat_" + col
  df1[cat_col] = cat_dummies[col]

In [11]:
df1.to_csv('data/merged_ai_descriptors_dummies_filtered.csv', index=False)